# Human-in-the-Loop: Pre-Action Gates, Livellamento del Rischio e Audit Logging

Il README di questa lezione introduce Human-in-the-Loop con un breve esempio che chiede all'utente di `APPROVARE` o `RIFIUTARE` dopo che l'agente ha già prodotto una risposta. Quel modello è un buon punto di partenza, ma le implementazioni HITL in produzione comunemente necessitano tre elementi aggiuntivi:

1. Un **gate pre-azione** che funziona **prima** che l'agente esegua un passo rischioso, così che costi, irreversibilità e latenza rimangano sotto controllo.
2. Il **livellamento del rischio**, così le azioni a basso rischio si eseguono automaticamente, quelle a rischio medio sono approvate in blocco, e solo le azioni ad alto rischio sono bloccate da un intervento umano.
3. Un **registro di audit più un ciclo di revisione**, così ogni decisione del gate è registrata come JSONL, e un rifiuto ri-inserisce l’agente con una motivazione strutturata invece di limitarsi a stampare `Revising...`.

Questo notebook costruisce ciascuno di questi elementi sopra le stesse primitive di `06-system-message-framework.ipynb`. Viene eseguito end-to-end in `DEMO_MODE = True` (non è necessario alcun input interattivo) o con veri prompt `input()` quando `DEMO_MODE = False`. Nota: in DEMO_MODE il ritentativo del terzo obiettivo è scritto in script, così la meccanica del ciclo è visibile end-to-end. La riclassificazione guidata dalla revisione vera richiede `DEMO_MODE = False` e un operatore.

**Fuori ambito (gestito in altre lezioni):** autenticazione e controllo accessi (minaccia 2 del README della Lezione 06), middleware per chiamate agli strumenti (approfondimento MAF della Lezione 14), modelli di dibattito multi-agente.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Modello 1: Cancello pre-azione

Il frammento HITL del README chiama prima l'agente, poi chiede all'utente di approvare il risultato. Questo è un flusso **post-azione**. L'agente ha già eseguito, quindi il costo della chiamata LLM è già stato pagato, e qualsiasi effetto collaterale (email inviata, riga di database scritta, commento pubblicato) è già avvenuto.

Un flusso **pre-azione** inserisce il cancello prima che l'agente esegua il passo rischioso. L'agente propone l'azione, il cancello decide se eseguirla, e solo con l'approvazione l'effetto collaterale si verifica.

| Aspetto | Approvazione post-azione (frammento README) | Cancello pre-azione (questo notebook) |
|---|---|---|
| Quando avviene l'approvazione? | Dopo che l'agente ha prodotto output | Prima che qualsiasi effetto collaterale venga eseguito |
| Costo LLM in caso di rifiuto | Già pagato | Pagato solo per la proposta, non per l'azione |
| Effetti collaterali irreversibili | Possibili (l'azione è già avvenuta) | Previsti |
| Chiarezza audit | L'approvazione è una stampa a video | L'approvazione è un record JSON con timestamp, azione, motivo |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Modello 2: Livellazione del rischio

Non ogni azione necessita di approvazione umana. Una ricerca in sola lettura contro un'API pubblica ha rischi diversi rispetto all'invio di un'email a un cliente. Trattare entrambi allo stesso modo spreca l'attenzione dell'operatore e rallenta l'agente.

Un modello semplice a 3 livelli:

| Tier | Esempi | Flusso di approvazione |
|---|---|---|
| `low` (sola lettura) | Ricerca in una knowledge base, consultare opzioni di volo, recuperare una pagina web pubblica | Esecuzione automatica, registrata per audit |
| `medium` (mutazione economica) | Memorizzare una cache, incrementare un contatore, programmare un promemoria | Esecuzione automatica, ma revisione giornaliera in batch |
| `high` (rivolto all'esterno o irreversibile) | Inviare un'email, addebitare una carta, pubblicare in un canale pubblico | Blocco in attesa di approvazione umana |

Questa è una forma di livellamento. I sistemi di produzione spesso usano livelli più granulari (ad esempio, livelli di autorizzazione IAM AWS, livelli di accesso basati sui ruoli). La versione a 3 livelli qui sotto è la versione minima utile per un agente che combina azioni di sola lettura e con effetti collaterali.

Il classificatore qui sotto usa euristiche basate su parole chiave affinché la demo rimanga deterministica ed economica. In un sistema di produzione si sostituirebbe con un classificatore appreso o un motore di policy.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Modello 3: Registro di controllo e ciclo di revisione

Un `print("Response approved.")` non è un registro di controllo. Per fiducia, ogni decisione del gate dovrebbe essere registrata come evento strutturato che puoi poi interrogare, riprodurre o allegare a una revisione dell'incidente.

Due elementi:

1. **JSONL append-only.** Una riga per ogni decisione, con timestamp, azione, livello, decisione, motivo. Facile da cercare con grep, facile da inviare a un vero archivio di log successivamente.
2. **Ciclo di revisione in caso di rifiuto.** Quando il gate restituisce `deny`, l'agente si richiama con il motivo del rifiuto nel contesto, così la proposta successiva può evitare il problema.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Risorse aggiuntive

Altri progetti pubblici implementano diverse varianti di questi modelli HITL. Confronta gli approcci per trovare quello che si adatta al tuo stack:

- **LangChain** wrapper di strumenti human-in-the-loop ([docs](https://python.langchain.com/docs/integrations/tools/human_tools)): wrapper di strumenti plug-and-play che fermano l'esecuzione per l'input umano.
- **AutoGen** `UserProxyAgent` ([v0.2 docs](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ ha ristrutturato questo): utilizza un ruolo agente specificamente per rappresentare l'umano nelle conversazioni multi-agente.
- **Microsoft Agent Framework (MAF)** middleware di invocazione di funzione ([docs](https://learn.microsoft.com/agent-framework/)): middleware che viene eseguito attorno a ogni chiamata di strumento/funzione, adatto per logica di controllo e flussi di approvazione.

Ogni progetto gestisce i tre sotto-modelli in modo diverso: LangChain li avvolge come strumenti, AutoGen usa un ruolo agente e Microsoft Agent Framework usa middleware di invocazione di funzione. Leggi una o due implementazioni integralmente prima di scegliere un design per il tuo agente.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
